# ShinkaEvolve Ansatz Search for Tic-Tac-Toe QML

This notebook prepares a ShinkaEvolve experiment to search for a 9-qubit ansatz for tic-tac-toe classification with a simulated variational quantum learning model.

It does not launch the ShinkaEvolve loop by default.  The launch cell is guarded by `RUN_EVOLUTION = False`; change that flag only after checking the generated files and environment.

The fixed model uses:

- 9 qubits, one per tic-tac-toe board cell.
- Paper-style board encoding: cross = `+1`, circle = `-1`, empty = `0`.
- Pauli-X feature map angles `2*pi/3 * cell_value`.
- Data re-uploading with `l = 3`.
- Two ansatz repetitions after each upload, `p = 2`.
- Simulated analytic PennyLane device: `default.qubit`, `shots=None`.
- Labels as `+1` for the correct class and `-1` for the others.


## Reference Setup

The symmetry paper enumerates legal tic-tac-toe games, includes unfinished games, labels unfinished games as draw, and samples class-balanced train/test sets.  It uses 450 training games, coming from 30 steps with 15 games per step, and 600 test games.

This notebook keeps those sizes and adds a 300-game validation split.  Because there are fewer unique circle-winning board states than the sum needed for strict disjoint 450/300/600 class-balanced unique splits, the default sampler follows the paper's random class-balanced protocol with replacement.  The notebook logs unique counts and split overlaps so this choice is visible.

Paper qubit indexing is used:

```text
0 -- 1 -- 2
|    |    |
7 -- 8 -- 3
|    |    |
6 -- 5 -- 4
```

The measured observables are reordered to the user-facing class order `[cross, circle, draw]`:

- `cross`: average Pauli-Z over edge qubits `(1, 3, 5, 7)`.
- `circle`: average Pauli-Z over corner qubits `(0, 2, 4, 6)`.
- `draw`: Pauli-Z on center qubit `8`.


## Required API Keys

This notebook is configured to use OpenRouter model names for ShinkaEvolve proposals.

Required for the provided config:

```bash
OPENROUTER_API_KEY="<your-openrouter-key>"
```

Optional only if you change `LLM_MODELS`, `META_LLM_MODELS`, or `EMBEDDING_MODEL` to native providers:

```bash
OPENAI_API_KEY="<your-openai-key>"
ANTHROPIC_API_KEY="<your-anthropic-key>"
GOOGLE_API_KEY="<your-google-key>"
```

The local evaluator itself does not need an API key.  API keys are only used by ShinkaEvolve when the evolution loop is launched.


## Environment

Use a local virtual environment before running the notebook.  From the repository root:

```bash
python3 -m venv .venv-shinka-ttt
source .venv-shinka-ttt/bin/activate
python -m pip install --upgrade pip
python -m pip install shinka-evolve pennylane pennylane-lightning scipy scikit-learn pandas matplotlib python-dotenv
python -m ipykernel install --user --name shinka-ttt --display-name "Python (shinka-ttt)"
```

Then select the `Python (shinka-ttt)` kernel for this notebook.


In [1]:
# Optional in-notebook installation cell.
# Prefer running these commands in a terminal, then selecting the matching kernel.

# !python3 -m venv .venv-shinka-ttt
# !. .venv-shinka-ttt/bin/activate && python -m pip install --upgrade pip
# !. .venv-shinka-ttt/bin/activate && python -m pip install shinka-evolve pennylane pennylane-lightning scipy scikit-learn pandas matplotlib python-dotenv ipykernel
# !. .venv-shinka-ttt/bin/activate && python -m ipykernel install --user --name shinka-ttt --display-name "Python (shinka-ttt)"


In [2]:
import os
import sys
import json
import logging
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

try:
    import dotenv
    dotenv.load_dotenv()
except Exception as exc:
    print(f"python-dotenv not available yet: {exc}")

for logger_name in ("httpx", "openai", "anthropic", "httpcore"):
    logging.getLogger(logger_name).setLevel(logging.WARNING)

print(f"Python: {sys.executable}")
for key in ("OPENROUTER_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY"):
    print(f"{key}: {'set' if os.environ.get(key) else 'missing'}")


Python: /home/chengyou1368/QuantumAnsatz/tic-tac-toe/.venv-shinka-ttt/bin/python
OPENROUTER_API_KEY: set
OPENAI_API_KEY: missing
ANTHROPIC_API_KEY: missing
GOOGLE_API_KEY: missing


## Generate ShinkaEvolve Program Files

The next two cells write the fixed seed program and evaluator expected by ShinkaEvolve.

Only `ANSATZ_SPEC` is evolvable.  It is a formal list of gate dictionaries.  The LLM can propose a valid ansatz by editing this list, and the fixed validator rejects unsupported gates, non-grid two-qubit operations, bad wire indices, and invalid parameter keys.

Parameter sharing is explicit: if two rotation gates use the same `param` string inside one ansatz block, they share that angle inside that block.  The `l=3` uploads and `p=2` repetitions still receive independent copies of the block parameters.


In [ ]:
%%writefile initial_program.py
"""Seed program for ShinkaEvolve tic-tac-toe QML ansatz search.

Only ANSATZ_SPEC is inside the EVOLVE-BLOCK.  The data construction,
feature map, re-uploading layout, training loop, measurements, and metric
packaging are fixed so candidates are compared on the same task.
"""

from __future__ import annotations

import json
import os
from collections import defaultdict
from pathlib import Path

import numpy as np
import pennylane as qml
from pennylane import numpy as pnp


# Problem constants
N_QUBITS = 9
N_UPLOADS = 3
N_REPEATS = 2
FEATURE_SCALE = 2 * np.pi / 3

# Paper indexing:
#   0 -- 1 -- 2
#   |    |    |
#   7 -- 8 -- 3
#   |    |    |
#   6 -- 5 -- 4
PAPER_GRID = ((0, 1, 2), (7, 8, 3), (6, 5, 4))
CORNERS = (0, 2, 4, 6)
EDGES = (1, 3, 5, 7)
CENTER = 8
GRID_EDGES = (
    (0, 1), (1, 2), (2, 3), (3, 4),
    (4, 5), (5, 6), (6, 7), (7, 0),
    (1, 8), (3, 8), (5, 8), (7, 8),
)
GRID_EDGE_SET = {tuple(sorted(edge)) for edge in GRID_EDGES}
WIN_LINES = (
    (0, 1, 2), (7, 8, 3), (6, 5, 4),
    (0, 7, 6), (1, 8, 5), (2, 3, 4),
    (0, 8, 4), (2, 8, 6),
)

# User-facing class order.  Observables are mapped to this order below.
CLASS_NAMES = ("cross", "circle", "draw")
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
LABEL_VECTORS = {
    "cross": np.array([1.0, -1.0, -1.0]),
    "circle": np.array([-1.0, 1.0, -1.0]),
    "draw": np.array([-1.0, -1.0, 1.0]),
}

ALLOWED_SINGLE_QUBIT_GATES = {"RX", "RY", "RZ"}
ALLOWED_TWO_QUBIT_GATES = {"CNOT", "CZ"}
ALLOWED_PARAM_TWO_QUBIT_GATES = {"CRX", "CRY", "CRZ"}


# EVOLVE-BLOCK-START
ANSATZ_SPEC = [
    {"gate": "RY", "wire": 0, "param": "ry_0"},
    {"gate": "RY", "wire": 1, "param": "ry_1"},
    {"gate": "RY", "wire": 2, "param": "ry_2"},
    {"gate": "RY", "wire": 3, "param": "ry_3"},
    {"gate": "RY", "wire": 4, "param": "ry_4"},
    {"gate": "RY", "wire": 5, "param": "ry_5"},
    {"gate": "RY", "wire": 6, "param": "ry_6"},
    {"gate": "RY", "wire": 7, "param": "ry_7"},
    {"gate": "RY", "wire": 8, "param": "ry_8"},
    {"gate": "RZ", "wire": 0, "param": "rz_pre_0"},
    {"gate": "RZ", "wire": 1, "param": "rz_pre_1"},
    {"gate": "RZ", "wire": 2, "param": "rz_pre_2"},
    {"gate": "RZ", "wire": 3, "param": "rz_pre_3"},
    {"gate": "RZ", "wire": 4, "param": "rz_pre_4"},
    {"gate": "RZ", "wire": 5, "param": "rz_pre_5"},
    {"gate": "RZ", "wire": 6, "param": "rz_pre_6"},
    {"gate": "RZ", "wire": 7, "param": "rz_pre_7"},
    {"gate": "RZ", "wire": 8, "param": "rz_pre_8"},
    {"gate": "CZ", "wires": [0, 1]},
    {"gate": "CZ", "wires": [1, 2]},
    {"gate": "CZ", "wires": [2, 3]},
    {"gate": "CZ", "wires": [3, 4]},
    {"gate": "CZ", "wires": [4, 5]},
    {"gate": "CZ", "wires": [5, 6]},
    {"gate": "CZ", "wires": [6, 7]},
    {"gate": "CZ", "wires": [7, 0]},
    {"gate": "CZ", "wires": [1, 8]},
    {"gate": "CZ", "wires": [3, 8]},
    {"gate": "CZ", "wires": [5, 8]},
    {"gate": "CZ", "wires": [7, 8]},
    {"gate": "RZ", "wire": 0, "param": "rz_post_0"},
    {"gate": "RZ", "wire": 1, "param": "rz_post_1"},
    {"gate": "RZ", "wire": 2, "param": "rz_post_2"},
    {"gate": "RZ", "wire": 3, "param": "rz_post_3"},
    {"gate": "RZ", "wire": 4, "param": "rz_post_4"},
    {"gate": "RZ", "wire": 5, "param": "rz_post_5"},
    {"gate": "RZ", "wire": 6, "param": "rz_post_6"},
    {"gate": "RZ", "wire": 7, "param": "rz_post_7"},
    {"gate": "RZ", "wire": 8, "param": "rz_post_8"},
]
# EVOLVE-BLOCK-END


def board_winner(board: tuple[int, ...] | np.ndarray) -> str | None:
    """Return 'cross', 'circle', or None if no player has three in a row."""
    for line in WIN_LINES:
        total = int(sum(int(board[i]) for i in line))
        if total == 3:
            return "cross"
        if total == -3:
            return "circle"
    return None


def board_label(board: tuple[int, ...] | np.ndarray) -> str:
    """Unfinished and full non-winning boards are labeled as draw."""
    return board_winner(board) or "draw"


def enumerate_valid_boards() -> np.ndarray:
    """Enumerate unique legal tic-tac-toe board states.

    Cross moves first.  Recursion stops once a winner appears or the board is
    full, so post-win positions are excluded.  Intermediate non-terminal
    states are included and labeled as draw, matching the paper's setup.
    """
    seen: set[tuple[int, ...]] = set()

    def visit(board: tuple[int, ...], player: int) -> None:
        if board in seen:
            return
        seen.add(board)
        if board_winner(board) is not None or all(value != 0 for value in board):
            return
        for wire, value in enumerate(board):
            if value == 0:
                next_board = list(board)
                next_board[wire] = player
                visit(tuple(next_board), -player)

    visit((0,) * N_QUBITS, 1)
    return np.array(sorted(seen), dtype=np.int8)


def make_balanced_split(
    grouped_boards: dict[str, np.ndarray],
    size: int,
    rng: np.random.Generator,
    replace: bool,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Sample a class-balanced split with +/-1 one-hot labels."""
    if size % len(CLASS_NAMES) != 0:
        raise ValueError(f"split size {size} must be divisible by {len(CLASS_NAMES)}")
    per_class = size // len(CLASS_NAMES)
    xs = []
    ys = []
    labels = []
    for label in CLASS_NAMES:
        pool = grouped_boards[label]
        local_replace = replace or per_class > len(pool)
        indices = rng.choice(len(pool), size=per_class, replace=local_replace)
        xs.append(pool[indices])
        ys.append(np.tile(LABEL_VECTORS[label], (per_class, 1)))
        labels.extend([label] * per_class)
    x = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    label_array = np.array(labels)
    order = rng.permutation(len(x))
    return x[order], y[order], label_array[order]


def build_data_splits(
    seed: int = 2027,
    train_size: int = 450,
    validation_size: int = 300,
    test_size: int = 600,
    replace: bool = True,
) -> dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """Construct paper-style class-balanced train/validation/test data.

    The paper uses 450 training games and 600 test games, class-balanced across
    cross win, circle win, and draw.  There are only 316 unique circle-winning
    legal board states under this enumeration, so this implementation samples
    independently with replacement by default to keep the paper's sizes and add
    a validation split.  Set replace=False only when using smaller split sizes.
    """
    rng = np.random.default_rng(seed)
    boards = enumerate_valid_boards()
    grouped = {name: [] for name in CLASS_NAMES}
    for board in boards:
        grouped[board_label(board)].append(board)
    grouped_arrays = {
        name: np.array(values, dtype=np.int8)
        for name, values in grouped.items()
    }
    return {
        "train": make_balanced_split(grouped_arrays, train_size, rng, replace),
        "validation": make_balanced_split(grouped_arrays, validation_size, rng, replace),
        "test": make_balanced_split(grouped_arrays, test_size, rng, replace),
    }


def dataset_summary(splits: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]]) -> dict:
    """Return class counts, unique counts, and split overlaps for logging."""
    summary = {}
    board_sets = {}
    for split_name, (x, _y, labels) in splits.items():
        board_sets[split_name] = {tuple(row.tolist()) for row in x}
        class_counts = {
            label: int(np.sum(labels == label))
            for label in CLASS_NAMES
        }
        summary[split_name] = {
            "rows": int(len(x)),
            "unique_boards": int(len(board_sets[split_name])),
            "class_counts": class_counts,
        }
    for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
        summary[f"overlap_{left}_{right}"] = int(len(board_sets[left] & board_sets[right]))
    all_boards = enumerate_valid_boards()
    all_counts = defaultdict(int)
    for board in all_boards:
        all_counts[board_label(board)] += 1
    summary["all_valid_unique_boards"] = {
        "total": int(len(all_boards)),
        "class_counts": {label: int(all_counts[label]) for label in CLASS_NAMES},
    }
    return summary


def validate_ansatz_spec(spec: list[dict]) -> tuple[list[str], list[str]]:
    """Validate ANSATZ_SPEC and return errors plus ordered parameter keys."""
    errors = []
    parameter_keys = []
    if not isinstance(spec, list) or not spec:
        return ["ANSATZ_SPEC must be a non-empty list of gate dictionaries."], []

    for gate_index, item in enumerate(spec):
        prefix = f"gate {gate_index}"
        if not isinstance(item, dict):
            errors.append(f"{prefix}: expected a dict, got {type(item).__name__}")
            continue
        gate = str(item.get("gate", "")).upper()
        if gate in ALLOWED_SINGLE_QUBIT_GATES:
            wire = item.get("wire")
            if not isinstance(wire, int) or not 0 <= wire < N_QUBITS:
                errors.append(f"{prefix}: {gate} requires integer wire in [0, {N_QUBITS - 1}]")
            param = item.get("param")
            if not isinstance(param, str) or not param:
                errors.append(f"{prefix}: {gate} requires a non-empty string param key")
            else:
                parameter_keys.append(param)
        elif gate in ALLOWED_TWO_QUBIT_GATES | ALLOWED_PARAM_TWO_QUBIT_GATES:
            wires = item.get("wires")
            if (
                not isinstance(wires, (list, tuple))
                or len(wires) != 2
                or not all(isinstance(w, int) for w in wires)
                or wires[0] == wires[1]
                or not all(0 <= w < N_QUBITS for w in wires)
            ):
                errors.append(f"{prefix}: {gate} requires two distinct integer wires")
            elif tuple(sorted(wires)) not in GRID_EDGE_SET:
                errors.append(f"{prefix}: {gate} uses non-adjacent grid edge {wires}")
            if gate in ALLOWED_PARAM_TWO_QUBIT_GATES:
                param = item.get("param")
                if not isinstance(param, str) or not param:
                    errors.append(f"{prefix}: {gate} requires a non-empty string param key")
                else:
                    parameter_keys.append(param)
        else:
            allowed = sorted(ALLOWED_SINGLE_QUBIT_GATES | ALLOWED_TWO_QUBIT_GATES | ALLOWED_PARAM_TWO_QUBIT_GATES)
            errors.append(f"{prefix}: unsupported gate {gate!r}; allowed gates are {allowed}")

    ordered_unique_keys = list(dict.fromkeys(parameter_keys))
    return errors, ordered_unique_keys


SPEC_ERRORS, PARAMETER_KEYS_PER_BLOCK = validate_ansatz_spec(ANSATZ_SPEC)
N_PARAMS_PER_BLOCK = len(PARAMETER_KEYS_PER_BLOCK)
N_PARAMS = N_PARAMS_PER_BLOCK * N_UPLOADS * N_REPEATS


def feature_map(board: np.ndarray) -> None:
    """Encode cell values -1, 0, +1 as equidistant Pauli-X rotations."""
    for wire in range(N_QUBITS):
        qml.RX(float(FEATURE_SCALE * int(board[wire])), wires=wire)


def apply_ansatz_block(params, block_index: int) -> None:
    """Apply one candidate ansatz block from the formal ANSATZ_SPEC."""
    if SPEC_ERRORS:
        raise ValueError("; ".join(SPEC_ERRORS))

    offset = block_index * N_PARAMS_PER_BLOCK
    key_to_position = {key: i for i, key in enumerate(PARAMETER_KEYS_PER_BLOCK)}
    for item in ANSATZ_SPEC:
        gate = str(item["gate"]).upper()
        if gate in ALLOWED_SINGLE_QUBIT_GATES:
            angle = params[offset + key_to_position[item["param"]]]
            wire = int(item["wire"])
            if gate == "RX":
                qml.RX(angle, wires=wire)
            elif gate == "RY":
                qml.RY(angle, wires=wire)
            elif gate == "RZ":
                qml.RZ(angle, wires=wire)
        elif gate in ALLOWED_TWO_QUBIT_GATES:
            wires = [int(w) for w in item["wires"]]
            if gate == "CNOT":
                qml.CNOT(wires=wires)
            elif gate == "CZ":
                qml.CZ(wires=wires)
        elif gate in ALLOWED_PARAM_TWO_QUBIT_GATES:
            angle = params[offset + key_to_position[item["param"]]]
            wires = [int(w) for w in item["wires"]]
            if gate == "CRX":
                qml.CRX(angle, wires=wires)
            elif gate == "CRY":
                qml.CRY(angle, wires=wires)
            elif gate == "CRZ":
                qml.CRZ(angle, wires=wires)


_DEVICE = qml.device("default.qubit", wires=N_QUBITS, shots=None)


@qml.qnode(_DEVICE, interface="autograd", diff_method="backprop")
def circuit_z_expectations(board, params):
    """Full simulated circuit returning per-qubit Z expectations."""
    for upload_index in range(N_UPLOADS):
        feature_map(board)
        for repeat_index in range(N_REPEATS):
            block_index = upload_index * N_REPEATS + repeat_index
            apply_ansatz_block(params, block_index)
    return [qml.expval(qml.PauliZ(wire)) for wire in range(N_QUBITS)]


def class_expectations(board, params):
    """Map nine Z expectations to [cross, circle, draw] outputs."""
    z = pnp.stack(circuit_z_expectations(board, params))
    cross = (z[1] + z[3] + z[5] + z[7]) / 4.0
    circle = (z[0] + z[2] + z[4] + z[6]) / 4.0
    draw = z[8]
    return pnp.stack([cross, circle, draw])


def predict_batch(params, x_batch: np.ndarray):
    return pnp.stack([class_expectations(board, params) for board in x_batch])


def l2_loss(params, x_batch: np.ndarray, y_batch: np.ndarray):
    predictions = predict_batch(params, x_batch)
    targets = pnp.array(y_batch, requires_grad=False)
    return pnp.mean(pnp.sum((predictions - targets) ** 2, axis=1))


def evaluate_split(params, x: np.ndarray, y: np.ndarray) -> dict[str, float]:
    predictions = np.asarray(predict_batch(params, x), dtype=float)
    loss = float(np.mean(np.sum((predictions - y) ** 2, axis=1)))
    accuracy = float(np.mean(np.argmax(predictions, axis=1) == np.argmax(y, axis=1)))
    return {"accuracy": accuracy, "loss": loss}


def _batch_indices(order: np.ndarray, step: int, batch_size: int) -> np.ndarray:
    start = (step * batch_size) % len(order)
    end = start + batch_size
    if end <= len(order):
        return order[start:end]
    return np.concatenate([order[start:], order[: end - len(order)]])


def _log(record: dict, log_file: Path | None, verbose: bool) -> None:
    line = json.dumps(record, sort_keys=True)
    if verbose:
        print(line)
    if log_file is not None:
        with log_file.open("a", encoding="utf-8") as handle:
            handle.write(line + "\n")


def circuit_metadata(params) -> dict:
    with qml.queuing.AnnotatedQueue() as queue:
        for upload_index in range(N_UPLOADS):
            feature_map(np.zeros(N_QUBITS, dtype=np.int8))
            for repeat_index in range(N_REPEATS):
                block_index = upload_index * N_REPEATS + repeat_index
                apply_ansatz_block(params, block_index)
    tape = qml.tape.QuantumScript.from_queue(queue)
    operations = [(op.name, op.wires.tolist()) for op in tape.operations]
    depth = tape.graph.get_depth() if hasattr(tape.graph, "get_depth") else len(operations)
    return {
        "operations": operations,
        "depth": int(depth),
        "gate_count": int(len(operations)),
    }


def run_experiment(
    seed: int = 0,
    data_seed: int | None = None,
    n_epochs: int | None = None,
    steps_per_epoch: int | None = None,
    batch_size: int | None = None,
    learning_rate: float | None = None,
    validation_size: int | None = None,
    convergence_threshold: float | None = None,
    eval_every_epochs: int | None = None,
    verbose: bool | None = None,
) -> dict:
    """Train the candidate ansatz and return ShinkaEvolve metrics."""
    if SPEC_ERRORS:
        return {
            "spec_valid": False,
            "error": "; ".join(SPEC_ERRORS),
            "n_qubits": N_QUBITS,
            "n_uploads": N_UPLOADS,
            "n_repeats": N_REPEATS,
            "n_params": N_PARAMS,
            "operations": [],
            "train_accuracy": 0.0,
            "validation_accuracy": 0.0,
            "test_accuracy": 0.0,
            "train_loss": float("inf"),
            "validation_loss": float("inf"),
            "test_loss": float("inf"),
        }

    data_seed = int(os.environ.get("DATA_SEED", "2027") if data_seed is None else data_seed)
    n_epochs = int(os.environ.get("N_EPOCHS", "100") if n_epochs is None else n_epochs)
    steps_per_epoch = int(os.environ.get("STEPS_PER_EPOCH", "30") if steps_per_epoch is None else steps_per_epoch)
    batch_size = int(os.environ.get("BATCH_SIZE", "15") if batch_size is None else batch_size)
    learning_rate = float(os.environ.get("LEARNING_RATE", "0.03") if learning_rate is None else learning_rate)
    validation_size = int(os.environ.get("VALIDATION_SIZE", "300") if validation_size is None else validation_size)
    convergence_threshold = float(
        os.environ.get("CONVERGENCE_THRESHOLD", "0.90")
        if convergence_threshold is None
        else convergence_threshold
    )
    eval_every_epochs = int(os.environ.get("EVAL_EVERY_EPOCHS", "1") if eval_every_epochs is None else eval_every_epochs)
    verbose = bool(int(os.environ.get("VERBOSE_TRAINING", "1"))) if verbose is None else bool(verbose)

    train_size = int(os.environ.get("TRAIN_SIZE", "450"))
    test_size = int(os.environ.get("TEST_SIZE", "600"))
    replace = bool(int(os.environ.get("SAMPLE_WITH_REPLACEMENT", "1")))

    rng = np.random.default_rng(seed)
    splits = build_data_splits(
        seed=data_seed,
        train_size=train_size,
        validation_size=validation_size,
        test_size=test_size,
        replace=replace,
    )
    x_train, y_train, _labels_train = splits["train"]
    x_validation, y_validation, _labels_validation = splits["validation"]
    x_test, y_test, _labels_test = splits["test"]

    log_root = os.environ.get("TTT_LOG_DIR")
    log_file = None
    if log_root:
        Path(log_root).mkdir(parents=True, exist_ok=True)
        log_file = Path(log_root) / f"candidate_seed_{seed}.jsonl"
        if log_file.exists():
            log_file.unlink()

    params = pnp.array(rng.uniform(-0.05, 0.05, size=N_PARAMS), requires_grad=True)
    optimizer = qml.AdamOptimizer(stepsize=learning_rate)
    max_steps = n_epochs * steps_per_epoch
    global_step = 0
    convergence_epoch = None
    convergence_step = None
    history = []

    _log(
        {
            "event": "start",
            "seed": int(seed),
            "data_seed": int(data_seed),
            "n_params": int(N_PARAMS),
            "n_params_per_block": int(N_PARAMS_PER_BLOCK),
            "n_uploads": int(N_UPLOADS),
            "n_repeats": int(N_REPEATS),
            "n_epochs": int(n_epochs),
            "steps_per_epoch": int(steps_per_epoch),
            "batch_size": int(batch_size),
            "learning_rate": float(learning_rate),
            "dataset": dataset_summary(splits),
        },
        log_file,
        verbose,
    )

    for epoch in range(1, n_epochs + 1):
        order = rng.permutation(len(x_train))
        last_batch_loss = None
        for step in range(steps_per_epoch):
            batch_ids = _batch_indices(order, step, batch_size)
            batch_x = x_train[batch_ids]
            batch_y = y_train[batch_ids]
            params, last_batch_loss = optimizer.step_and_cost(
                lambda candidate_params: l2_loss(candidate_params, batch_x, batch_y),
                params,
            )
            global_step += 1

        if epoch == 1 or epoch == n_epochs or epoch % eval_every_epochs == 0:
            train_metrics = evaluate_split(params, x_train, y_train)
            validation_metrics = evaluate_split(params, x_validation, y_validation)
            test_metrics = evaluate_split(params, x_test, y_test)
            if convergence_epoch is None and validation_metrics["accuracy"] >= convergence_threshold:
                convergence_epoch = epoch
                convergence_step = global_step
            record = {
                "event": "epoch",
                "epoch": int(epoch),
                "global_step": int(global_step),
                "batch_loss": float(last_batch_loss),
                "train_accuracy": train_metrics["accuracy"],
                "validation_accuracy": validation_metrics["accuracy"],
                "test_accuracy": test_metrics["accuracy"],
                "train_loss": train_metrics["loss"],
                "validation_loss": validation_metrics["loss"],
                "test_loss": test_metrics["loss"],
            }
            history.append(record)
            _log(record, log_file, verbose)

    final_train = evaluate_split(params, x_train, y_train)
    final_validation = evaluate_split(params, x_validation, y_validation)
    final_test = evaluate_split(params, x_test, y_test)
    metadata = circuit_metadata(params)
    generalization_gap = abs(final_train["accuracy"] - final_test["accuracy"])
    parameter_efficiency = final_test["accuracy"] / max(float(N_PARAMS), 1.0)

    result = {
        "spec_valid": True,
        "n_qubits": N_QUBITS,
        "n_uploads": N_UPLOADS,
        "n_repeats": N_REPEATS,
        "n_params_per_block": N_PARAMS_PER_BLOCK,
        "n_params": N_PARAMS,
        "depth": metadata["depth"],
        "gate_count": metadata["gate_count"],
        "operations": metadata["operations"],
        "train_accuracy": final_train["accuracy"],
        "validation_accuracy": final_validation["accuracy"],
        "test_accuracy": final_test["accuracy"],
        "train_loss": final_train["loss"],
        "validation_loss": final_validation["loss"],
        "test_loss": final_test["loss"],
        "generalization_gap": generalization_gap,
        "parameter_efficiency": parameter_efficiency,
        "convergence_threshold": convergence_threshold,
        "convergence_epoch": convergence_epoch,
        "convergence_step": convergence_step,
        "max_steps": max_steps,
        "history_tail": history[-10:],
    }
    _log({"event": "final", **result}, log_file, verbose)
    return result


In [4]:
%%writefile evaluate.py
"""Evaluator for ShinkaEvolve tic-tac-toe QML ansatz search."""

from __future__ import annotations

import json
import math
import os
from functools import partial
from pathlib import Path

import numpy as np

from shinka.core import run_shinka_eval


NUM_RUNS = int(os.environ.get("NUM_RUNS", "2"))
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", "1"))
BASE_SEED = int(os.environ.get("BASE_SEED", "1000"))
DATA_SEED = int(os.environ.get("DATA_SEED", "2027"))

TRAIN_SIZE = int(os.environ.get("TRAIN_SIZE", "450"))
VALIDATION_SIZE = int(os.environ.get("VALIDATION_SIZE", "300"))
TEST_SIZE = int(os.environ.get("TEST_SIZE", "600"))
N_EPOCHS = int(os.environ.get("N_EPOCHS", "100"))
STEPS_PER_EPOCH = int(os.environ.get("STEPS_PER_EPOCH", "30"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "15"))
LEARNING_RATE = float(os.environ.get("LEARNING_RATE", "0.03"))
CONVERGENCE_THRESHOLD = float(os.environ.get("CONVERGENCE_THRESHOLD", "0.90"))
EVAL_EVERY_EPOCHS = int(os.environ.get("EVAL_EVERY_EPOCHS", "1"))

MAX_PARAMS = int(os.environ.get("MAX_PARAMS", "768"))
SEED_TOTAL_PARAMS = 162
USE_TEST_IN_SCORE = bool(int(os.environ.get("USE_TEST_IN_SCORE", "0")))
GRID_EDGE_SET = {
    tuple(sorted(edge))
    for edge in (
        (0, 1), (1, 2), (2, 3), (3, 4),
        (4, 5), (5, 6), (6, 7), (7, 0),
        (1, 8), (3, 8), (5, 8), (7, 8),
    )
}


def get_experiment_kwargs(run_index: int) -> dict:
    """Use the same data split and different optimization seeds per run."""
    return {
        "seed": BASE_SEED + run_index,
        "data_seed": DATA_SEED,
        "n_epochs": N_EPOCHS,
        "steps_per_epoch": STEPS_PER_EPOCH,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "validation_size": VALIDATION_SIZE,
        "convergence_threshold": CONVERGENCE_THRESHOLD,
        "eval_every_epochs": EVAL_EVERY_EPOCHS,
        "verbose": bool(int(os.environ.get("VERBOSE_TRAINING", "1"))),
    }


def _finite_number(value) -> bool:
    return isinstance(value, (int, float)) and math.isfinite(float(value))


def validate_fn(result) -> tuple[bool, str | None]:
    if not isinstance(result, dict):
        return False, f"Expected dict result, got {type(result).__name__}"
    if not result.get("spec_valid", False):
        return False, result.get("error", "ANSATZ_SPEC is invalid")
    if result.get("n_qubits") != 9:
        return False, f"Expected 9 qubits, got {result.get('n_qubits')}"
    if result.get("n_uploads") != 3 or result.get("n_repeats") != 2:
        return False, "The fixed architecture must keep l=3 and p=2"
    n_params = result.get("n_params")
    if not isinstance(n_params, int) or n_params <= 0 or n_params > MAX_PARAMS:
        return False, f"Invalid parameter count: {n_params}"

    for key in (
        "train_accuracy", "validation_accuracy", "test_accuracy",
        "train_loss", "validation_loss", "test_loss",
        "generalization_gap", "parameter_efficiency",
    ):
        if not _finite_number(result.get(key)):
            return False, f"{key} is missing or non-finite: {result.get(key)}"
    for key in ("train_accuracy", "validation_accuracy", "test_accuracy"):
        value = float(result[key])
        if not 0.0 <= value <= 1.0:
            return False, f"{key} is outside [0, 1]: {value}"

    for gate_name, wires in result.get("operations", []):
        if len(wires) == 2 and tuple(sorted(wires)) not in GRID_EDGE_SET:
            return False, f"Connectivity violation: {gate_name} on wires {wires}"
    return True, None


def score_result(result: dict) -> dict:
    """Compute normalized score components for one training seed."""
    primary_acc_key = "test_accuracy" if USE_TEST_IN_SCORE else "validation_accuracy"
    primary_loss_key = "test_loss" if USE_TEST_IN_SCORE else "validation_loss"
    primary_accuracy = float(result[primary_acc_key])
    primary_loss = float(result[primary_loss_key])
    train_accuracy = float(result["train_accuracy"])
    test_accuracy = float(result["test_accuracy"])
    gap = abs(train_accuracy - test_accuracy)
    n_params = max(float(result["n_params"]), 1.0)
    max_steps = max(float(result.get("max_steps") or (N_EPOCHS * STEPS_PER_EPOCH)), 1.0)
    convergence_step = result.get("convergence_step")

    gap_score = max(0.0, 1.0 - min(gap / 0.35, 1.0))
    loss_score = 1.0 / (1.0 + max(primary_loss, 0.0))
    parameter_efficiency_score = min(1.0, primary_accuracy * SEED_TOTAL_PARAMS / n_params)
    convergence_score = 0.0
    if convergence_step is not None:
        convergence_score = max(0.0, 1.0 - min(float(convergence_step) / max_steps, 1.0))

    combined_score = (
        0.50 * primary_accuracy
        + 0.10 * train_accuracy
        + 0.15 * gap_score
        + 0.15 * loss_score
        + 0.05 * parameter_efficiency_score
        + 0.05 * convergence_score
    )
    return {
        "combined_score": float(combined_score),
        "primary_accuracy": primary_accuracy,
        "gap_score": gap_score,
        "loss_score": loss_score,
        "parameter_efficiency_score": parameter_efficiency_score,
        "convergence_score": convergence_score,
    }


def _mean(results: list[dict], key: str) -> float:
    return float(np.mean([float(result[key]) for result in results]))


def _std(results: list[dict], key: str) -> float:
    return float(np.std([float(result[key]) for result in results]))


def aggregate_metrics(results: list[dict], results_dir: str) -> dict:
    scored = [score_result(result) for result in results]
    combined_score = float(np.mean([item["combined_score"] for item in scored]))

    public = {
        "train_accuracy_mean": round(_mean(results, "train_accuracy"), 4),
        "train_accuracy_std": round(_std(results, "train_accuracy"), 4),
        "validation_accuracy_mean": round(_mean(results, "validation_accuracy"), 4),
        "validation_accuracy_std": round(_std(results, "validation_accuracy"), 4),
        "test_accuracy_mean": round(_mean(results, "test_accuracy"), 4),
        "test_accuracy_std": round(_std(results, "test_accuracy"), 4),
        "generalization_gap_mean": round(_mean(results, "generalization_gap"), 4),
        "validation_loss_mean": round(_mean(results, "validation_loss"), 4),
        "test_loss_mean": round(_mean(results, "test_loss"), 4),
        "parameter_efficiency_mean": round(_mean(results, "parameter_efficiency"), 6),
        "n_params": int(results[0]["n_params"]),
        "depth_mean": round(_mean(results, "depth"), 1),
        "gate_count_mean": round(_mean(results, "gate_count"), 1),
        "score_uses_test": USE_TEST_IN_SCORE,
    }

    convergence_steps = [
        result.get("convergence_step")
        for result in results
        if result.get("convergence_step") is not None
    ]
    public["convergence_step_mean"] = (
        round(float(np.mean(convergence_steps)), 1)
        if convergence_steps
        else None
    )

    results_path = Path(results_dir)
    results_path.mkdir(parents=True, exist_ok=True)
    with (results_path / "per_run_metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, indent=2)
    with (results_path / "score_components.json").open("w", encoding="utf-8") as handle:
        json.dump(scored, handle, indent=2)

    lines = [
        f"Combined score: {combined_score:.4f}",
        f"Validation accuracy mean: {public['validation_accuracy_mean']:.4f}",
        f"Test accuracy mean: {public['test_accuracy_mean']:.4f}",
        f"Train-test generalization gap mean: {public['generalization_gap_mean']:.4f}",
        f"Validation L2 loss mean: {public['validation_loss_mean']:.4f}",
        f"Parameters: {public['n_params']}, depth mean: {public['depth_mean']}, gate count mean: {public['gate_count_mean']}",
    ]
    if not USE_TEST_IN_SCORE:
        lines.append("Fitness uses validation metrics; test metrics are reported as holdout diagnostics.")
    if public["validation_accuracy_mean"] < 0.55:
        lines.append("Accuracy is still low; explore more expressive local entanglement or rotation diversity.")
    if public["generalization_gap_mean"] > 0.15:
        lines.append("Large train-test gap; prefer parameter sharing, fewer parameters, or symmetry-inspired repeated motifs.")
    if public["n_params"] > SEED_TOTAL_PARAMS:
        lines.append("Parameter count exceeds the Pauli-2 seed; added accuracy must justify the complexity.")
    if public["convergence_step_mean"] is None:
        lines.append(f"Did not reach convergence threshold {CONVERGENCE_THRESHOLD:.2f} in these runs.")

    return {
        "combined_score": combined_score,
        "public": public,
        "private": {
            "per_run_scores": scored,
            "per_run_test_accuracies": [float(result["test_accuracy"]) for result in results],
        },
        "text_feedback": "\n".join(lines),
    }


def main(program_path: str, results_dir: str) -> None:
    metrics, correct, error_msg = run_shinka_eval(
        program_path=program_path,
        results_dir=results_dir,
        experiment_fn_name="run_experiment",
        num_runs=NUM_RUNS,
        get_experiment_kwargs=get_experiment_kwargs,
        validate_fn=validate_fn,
        aggregate_metrics_fn=partial(aggregate_metrics, results_dir=results_dir),
        run_workers=NUM_WORKERS,
    )
    print("OK" if correct else f"FAILED: {error_msg}")
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser()
    parser.add_argument("--program_path", default="initial_program.py")
    parser.add_argument("--results_dir", default="results_test")
    args = parser.parse_args()

    os.makedirs(args.results_dir, exist_ok=True)
    main(args.program_path, args.results_dir)


Overwriting evaluate.py


## Inspect Data Construction

Run this after the program files are written.  It verifies the legal-board enumeration, class balance, unique counts, and overlap caused by replacement sampling.


In [5]:
import importlib
import json
import initial_program

importlib.reload(initial_program)

splits = initial_program.build_data_splits(
    seed=2027,
    train_size=450,
    validation_size=300,
    test_size=600,
    replace=True,
)
print(json.dumps(initial_program.dataset_summary(splits), indent=2))


{
  "train": {
    "rows": 450,
    "unique_boards": 404,
    "class_counts": {
      "cross": 150,
      "circle": 150,
      "draw": 150
    }
  },
  "validation": {
    "rows": 300,
    "unique_boards": 281,
    "class_counts": {
      "cross": 100,
      "circle": 100,
      "draw": 100
    }
  },
  "test": {
    "rows": 600,
    "unique_boards": 510,
    "class_counts": {
      "cross": 200,
      "circle": 200,
      "draw": 200
    }
  },
  "overlap_train_validation": 58,
  "overlap_train_test": 95,
  "overlap_validation_test": 77,
  "all_valid_unique_boards": {
    "total": 5478,
    "class_counts": {
      "cross": 626,
      "circle": 316,
      "draw": 4536
    }
  }
}


## Inspect the Seed Ansatz

The seed is an EfficientSU2-style RY/RZ/CZ block:

1. Independent `RY` rotations on every qubit.
2. Independent pre-entanglement `RZ` rotations on every qubit.
3. `CZ` on every adjacent grid edge.
4. Independent post-entanglement `RZ` rotations on every qubit.

With `l=3` and `p=2`, the seed has `3 * 3 * 2 = 18` trainable parameters.


In [ ]:
import initial_program

print(f"Spec errors: {initial_program.SPEC_ERRORS}")
print(f"Parameters per ansatz block: {initial_program.N_PARAMS_PER_BLOCK}")
print(f"Total trainable parameters: {initial_program.N_PARAMS}")
print(f"Number of gates in ANSATZ_SPEC: {len(initial_program.ANSATZ_SPEC)}")
print("First five gates:")
for gate in initial_program.ANSATZ_SPEC[:5]:
    print(gate)


In [ ]:
# Optional: draw the full seed circuit.
# This requires PennyLane and matplotlib and can be visually dense because l=3 and p=2.

import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt

@qml.qnode(qml.device("default.qubit", wires=initial_program.N_QUBITS))
def draw_seed(board, params):
    for upload_index in range(initial_program.N_UPLOADS):
        initial_program.feature_map(board)
        for repeat_index in range(initial_program.N_REPEATS):
            block_index = upload_index * initial_program.N_REPEATS + repeat_index
            initial_program.apply_ansatz_block(params, block_index)
    return [qml.expval(qml.PauliZ(wire)) for wire in range(initial_program.N_QUBITS)]

fig, ax = qml.draw_mpl(draw_seed, decimals=None)(
    np.zeros(initial_program.N_QUBITS, dtype=np.int8),
    np.zeros(initial_program.N_PARAMS),
)
ax.set_title(f"EfficientSU2-style seed ansatz, l=3, p=2, params={initial_program.N_PARAMS}", fontsize=10)
fig.set_size_inches(16, 5)
fig.tight_layout()
plt.show()


## Fast Smoke Test

This is not the ShinkaEvolve loop.  It trains the seed for one epoch and one step to catch wiring, differentiation, and metric-shape mistakes.  The real evaluator defaults remain paper-scale: 100 epochs, 30 steps per epoch, batch size 15.


In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    result = initial_program.run_experiment(
        seed=0,
        data_seed=2027,
        n_epochs=1,
        steps_per_epoch=1,
        batch_size=3,
        validation_size=30,
        learning_rate=0.01,
        eval_every_epochs=1,
        verbose=False,
    )
    evaluate_module = importlib.import_module("evaluate")
    valid, message = evaluate_module.validate_fn(result)
    score = evaluate_module.score_result(result) if valid else None
    print("valid:", valid, message)
    print(json.dumps({
        "combined_score": None if score is None else score["combined_score"],
        "train_accuracy": result["train_accuracy"],
        "validation_accuracy": result["validation_accuracy"],
        "test_accuracy": result["test_accuracy"],
        "generalization_gap": result["generalization_gap"],
        "n_params": result["n_params"],
        "depth": result["depth"],
        "gate_count": result["gate_count"],
    }, indent=2))
else:
    print("Smoke test skipped.")


## Scoring Function for ShinkaEvolve

Each candidate is trained with Adam and evaluated on:

- classification accuracy on train, validation, and test splits;
- empirical train-test generalization gap;
- L2 loss;
- parameter efficiency;
- convergence speed, measured by the first epoch/step reaching a validation accuracy threshold.

The default `combined_score` uses validation accuracy and validation loss for model selection, while test metrics are reported as holdout diagnostics.  Set `USE_TEST_IN_SCORE=1` only if you deliberately want the test metrics to affect ShinkaEvolve fitness.

Score weights in `evaluate.py`:

```text
0.50 * primary_accuracy
+ 0.10 * train_accuracy
+ 0.15 * gap_score
+ 0.15 * loss_score
+ 0.05 * parameter_efficiency_score
+ 0.05 * convergence_score
```


## Configure ShinkaEvolve

The next cell prepares a 10-generation ShinkaEvolve run but does not launch it.  Model names are OpenRouter-style names and can be replaced with models available to your account.


In [ ]:
import datetime as dt
import os
import shlex
import sys
from pathlib import Path

from shinka.core import EvolutionConfig, ShinkaEvolveRunner
from shinka.database import DatabaseConfig
from shinka.launch import LocalJobConfig

TASK_SYS_MSG = """
You are an expert in quantum machine learning and variational quantum circuits.

Goal:
Evolve the ANSATZ_SPEC for a 9-qubit tic-tac-toe classifier.  The fixed code
will train each candidate with Adam in a simulated PennyLane circuit.

Fixed architecture:
- 9 qubits using the paper's tic-tac-toe indexing.
- Feature map: RX(2*pi/3 * cell_value) on every qubit.
- Data re-uploading l=3.
- The candidate ansatz block is applied p=2 times after each re-uploading.
- The block receives independent parameter copies for each upload/repetition.
- Measurements produce [cross, circle, draw] expectation vectors.

Only edit the EVOLVE-BLOCK, especially ANSATZ_SPEC.  Do not change the fixed
training loop, data construction, measurements, l, p, or feature map.

Formal ANSATZ_SPEC schema:
- Single-qubit parametrized gates:
  {"gate": "RX"|"RY"|"RZ", "wire": int 0..8, "param": "name"}
- Fixed two-qubit gates:
  {"gate": "CNOT"|"CZ", "wires": [control_or_first, target_or_second]}
- Parametrized controlled rotations:
  {"gate": "CRX"|"CRY"|"CRZ", "wires": [control, target], "param": "name"}

Connectivity constraint:
Two-qubit gates are allowed only on adjacent grid edges:
(0,1), (1,2), (2,3), (3,4), (4,5), (5,6), (6,7), (7,0),
(1,8), (3,8), (5,8), (7,8).

Parameter sharing:
Reusing the same param string shares that parameter within one ansatz block.
Use sharing deliberately for parameter efficiency or symmetry-inspired motifs.

Starting point:
The seed is an EfficientSU2-style block: independent RY rotations on every
qubit, independent pre-entanglement RZ rotations, CZ gates on each adjacent
grid edge, and independent post-entanglement RZ rotations.

Candidate quality:
Improve validation/test accuracy, reduce the train-test gap, reduce L2 loss,
use parameters efficiently, and reach high accuracy in fewer Adam steps.

Invalid candidates will be rejected if they use unsupported gates, bad wires,
non-adjacent two-qubit operations, non-finite metrics, or too many parameters.
"""

# Quick evaluator defaults for interactive evolution.  Set EVAL_PRESET = "full"
# only when you intentionally want the long paper-scale run.
COMMON_EVAL_ENV = {
    "PYTHONUNBUFFERED": "1",
    "PYTHONIOENCODING": "utf-8",
    "MPLCONFIGDIR": "/tmp/matplotlib-shinka-ttt",
    "NUM_WORKERS": "1",
    "SAMPLE_WITH_REPLACEMENT": "1",
    "USE_TEST_IN_SCORE": "0",
    "TTT_LOG_DIR": "logs/ttt_training",
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
}
EVAL_PRESETS = {
    "quick": {
        "NUM_RUNS": "1",
        "TRAIN_SIZE": "60",
        "VALIDATION_SIZE": "45",
        "TEST_SIZE": "60",
        "N_EPOCHS": "3",
        "STEPS_PER_EPOCH": "3",
        "BATCH_SIZE": "10",
        "LEARNING_RATE": "0.03",
        "CONVERGENCE_THRESHOLD": "0.90",
        "EVAL_EVERY_EPOCHS": "1",
        "VERBOSE_TRAINING": "1",
    },
    "full": {
        "NUM_RUNS": "2",
        "TRAIN_SIZE": "450",
        "VALIDATION_SIZE": "300",
        "TEST_SIZE": "600",
        "N_EPOCHS": "100",
        "STEPS_PER_EPOCH": "30",
        "BATCH_SIZE": "15",
        "LEARNING_RATE": "0.03",
        "CONVERGENCE_THRESHOLD": "0.90",
        "EVAL_EVERY_EPOCHS": "1",
        "VERBOSE_TRAINING": "1",
    },
}
EVAL_PRESET = "quick"
if EVAL_PRESET not in EVAL_PRESETS:
    raise ValueError(f"Unknown EVAL_PRESET: {EVAL_PRESET}")
EVAL_SETTINGS = {**COMMON_EVAL_ENV, **EVAL_PRESETS[EVAL_PRESET]}
for key, value in EVAL_SETTINGS.items():
    os.environ[key] = value

experiment_name = "ttt_qml_" + dt.datetime.now().strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = "results/" + experiment_name
EVAL_PROGRAM_PATH = "evaluate.py"

LLM_MODELS = [
    "openrouter/anthropic/claude-haiku-4-5",
    "openrouter/openai/gpt-5-nano",
]
META_LLM_MODELS = ["openrouter/openai/o4-mini"]
NOVELTY_LLM_MODELS = ["openrouter/openai/o4-mini"]
EMBEDDING_MODEL = "openrouter/openai/text-embedding-3-small"

evo_config = EvolutionConfig(
    task_sys_msg=TASK_SYS_MSG,
    init_program_path="initial_program.py",
    results_dir=RESULTS_DIR,
    language="python",
    job_type="local",
    num_generations=20,
    max_api_costs=3.0,
    patch_types=["diff", "full", "cross"],
    patch_type_probs=[0.65, 0.25, 0.10],
    max_patch_resamples=3,
    max_patch_attempts=2,
    llm_models=LLM_MODELS,
    llm_kwargs={"temperatures": [0.0, 0.5, 1.0], "max_tokens": 16384},
    llm_dynamic_selection="ucb1",
    llm_dynamic_selection_kwargs={"exploration_coef": 1.0, "cost_aware_coef": 0.7},
    meta_rec_interval=5,
    meta_llm_models=META_LLM_MODELS,
    meta_llm_kwargs={"temperatures": [0.0], "max_tokens": 8192},
    embedding_model=EMBEDDING_MODEL,
    max_novelty_attempts=2,
    code_embed_sim_threshold=0.99,
    novelty_llm_models=NOVELTY_LLM_MODELS,
    novelty_llm_kwargs={"temperatures": [0.0]},
)

db_config = DatabaseConfig(
    num_islands=1,
    archive_size=20,
    elite_selection_ratio=0.30,
    num_archive_inspirations=1,
    num_top_k_inspirations=1,
    parent_selection_strategy="weighted",
    parent_selection_lambda=10,
    archive_selection_strategy="crowding",
    archive_criteria={
        "combined_score": 1.0,
        "validation_accuracy_mean": 0.5,
        "generalization_gap_mean": -0.3,
        "n_params": -0.1,
    },
    enable_dynamic_islands=False,
)

repo_activate_path = Path(".venv-shinka-ttt/bin/activate").resolve()
kernel_activate_path = Path(sys.executable).resolve().parent / "activate"
venv_activate_path = repo_activate_path if repo_activate_path.exists() else kernel_activate_path
if not venv_activate_path.exists():
    raise FileNotFoundError(f"Activation script not found: {venv_activate_path}")

eval_activate_path = Path("shinka_cli_task/activate_eval_env.sh").resolve()
eval_activate_path.parent.mkdir(parents=True, exist_ok=True)
activate_lines = [
    "#!/usr/bin/env bash\n",
    "set -e\n",
    f"source {shlex.quote(str(venv_activate_path))}\n",
]
for key in sorted(EVAL_SETTINGS):
    activate_lines.append(f'export {key}="${{{key}:-{EVAL_SETTINGS[key]}}}"\n')
eval_activate_path.write_text("".join(activate_lines), encoding="utf-8")
eval_activate_path.chmod(0o755)
activate_path = str(eval_activate_path)

job_config = LocalJobConfig(
    eval_program_path=EVAL_PROGRAM_PATH,
    activate_script=activate_path,
    time="02:00:00",
)

print(f"Experiment: {experiment_name}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Generations: {evo_config.num_generations}")
print(f"Models: {LLM_MODELS}")
print(f"Evaluator: {EVAL_PROGRAM_PATH}")
print(f"Evaluation preset: {EVAL_PRESET}")
print(f"Activation script: {activate_path}")


## Launch Guard

This cell intentionally does not run the evolution loop unless you change `RUN_EVOLUTION` to `True`.


In [ ]:
from time import perf_counter

runner = ShinkaEvolveRunner(
    evo_config=evo_config,
    job_config=job_config,
    db_config=db_config,
    max_evaluation_jobs=1,
    max_proposal_jobs=1,
    max_db_workers=2,
    verbose=True,
)

RUN_EVOLUTION = False

if RUN_EVOLUTION:
    tic = perf_counter()
    await runner.run_async()
    toc = perf_counter()
    print(f"Evolution completed in {toc - tic:.1f} s")
else:
    print("Evolution is configured but not launched. Set RUN_EVOLUTION = True when ready.")


## Inspect Results After a Run

Run these cells only after launching ShinkaEvolve.  They load candidate metrics, plot progress, and show the best evolved `ANSATZ_SPEC`.


In [ ]:
from pathlib import Path
import pandas as pd

from shinka.utils import load_programs_to_df

results_root = Path(RESULTS_DIR)
if results_root.exists():
    df = load_programs_to_df(str(results_root))
    print(f"Candidates evaluated: {len(df)}")
    if len(df):
        print(f"Best combined score: {df['combined_score'].max():.4f}")
        display(df.sort_values("combined_score", ascending=False).head(10))
else:
    print(f"{results_root} does not exist yet.")


In [ ]:
import matplotlib.pyplot as plt
from shinka.plots import plot_evals_performance, plot_lineage_tree

if "df" in globals() and len(df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_evals_performance(df, title="Tic-Tac-Toe QML Ansatz Search", fig=fig, ax=axes[0])
    plot_lineage_tree(df, title="Lineage Tree", fig=fig, ax=axes[1])
    plt.tight_layout()
    plt.show()
else:
    print("No results dataframe loaded.")


In [ ]:
import importlib.util
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt

best_py = Path(RESULTS_DIR) / "best" / "main.py"
if best_py.exists():
    source = best_py.read_text(encoding="utf-8")
    start = source.find("# EVOLVE-BLOCK-START")
    end = source.find("# EVOLVE-BLOCK-END") + len("# EVOLVE-BLOCK-END")
    print(source[start:end] if start >= 0 and end > start else source)

    spec = importlib.util.spec_from_file_location("best_program", best_py)
    best_program = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(best_program)

    @qml.qnode(qml.device("default.qubit", wires=best_program.N_QUBITS))
    def draw_best(board, params):
        for upload_index in range(best_program.N_UPLOADS):
            best_program.feature_map(board)
            for repeat_index in range(best_program.N_REPEATS):
                block_index = upload_index * best_program.N_REPEATS + repeat_index
                best_program.apply_ansatz_block(params, block_index)
        return [qml.expval(qml.PauliZ(wire)) for wire in range(best_program.N_QUBITS)]

    fig, ax = qml.draw_mpl(draw_best, decimals=None)(
        np.zeros(best_program.N_QUBITS, dtype=np.int8),
        np.zeros(best_program.N_PARAMS),
    )
    ax.set_title(f"Best evolved ansatz, params={best_program.N_PARAMS}", fontsize=10)
    fig.set_size_inches(16, 5)
    fig.tight_layout()
    plt.show()
else:
    print(f"{best_py} not found yet.")


## Notes for Real Runs

- The default evaluator is intentionally expensive because it matches the paper-scale optimization protocol.  For development, lower `N_EPOCHS`, `STEPS_PER_EPOCH`, `NUM_RUNS`, and `VALIDATION_SIZE`.
- Keep `USE_TEST_IN_SCORE=0` for honest model selection.  Use the test metrics for final reporting.
- The formal ansatz schema lets the LLM explore common hardware-efficient structures while keeping the 9-qubit data re-uploading circuit and scoring protocol fixed.
- The paper's invariant ansatz is not hard-coded as the seed.  The run starts from the simple Pauli-2 ansatz and lets ShinkaEvolve discover improvements, including possible parameter sharing and symmetry-inspired motifs.
